# ⏱️ 09_teste_tempo_real — Um arquivo novo atravessando as três camadas
**Squad 2 · Dupla 3** · teste do polling e do controle de arquivos, numa janela de tempo real (gerador ligado)

**Como usar**

| Quando | O que rodar |
|---|---|
| Antes de iniciar o orquestrador | Etapas 1 e 2 (grava o marco zero em `metadata/`) |
| Durante a janela | `06_orquestrador` |
| Depois que o orquestrador parar | Etapa 1 e Etapas 3 a 6 |

| Etapa | Pergunta |
|---|---|
| 3 | Algum ciclo recebeu arquivo novo na Bronze e, **no mesmo ciclo**, rodou Silver e Gold com sucesso? |
| 4 | Cada arquivo novo chegou inteiro à Silver ou à quarentena? Com que latência? |
| 5 | A Gold recalculou os indicadores depois da chegada desses pedidos? |
| 6 | Veredito e latência de ponta a ponta |

Somente leitura, exceto o marco zero (um JSON em `metadata/`). Cada etapa **faz** e **valida**: ✅ ok · ⚠️ achado (segue) · ❌ erro (para).

## Etapa 1 — Configuração
**Faz:** carrega o `00_setup_config` em modo rápido e o fuso das datas do gerador (contrato da EDA), usado para medir a latência em UTC.
**Valida:** se a Bronze existe.

In [0]:
VALIDACAO_COMPLETA = False

In [0]:
%run ./00_setup_config

# ⚙️ 00_setup_config — Configuração central
**Squad 2 · Dupla 3 · `ecommerce_pedidos`**

Prepara endereços, credenciais, conexões e funções usadas por todo o pipeline. **Não lê nem grava pedidos.**

Cada etapa **faz o trabalho e valida em seguida**: ✅ ok · ⚠️ aviso (segue) · ❌ erro (para e explica).

**Uso nos outros notebooks** (duas células; o `%run` fica sozinho):
```
VALIDACAO_COMPLETA = False
```
```
%run ./00_setup_config
```
Dependências em **Environment → Dependencies**: `azure-identity`, `azure-storage-file-datalake`.

## Etapa 1 — Ambiente e endereços
**Faz:** cria a função `checar` e guarda os nomes fixos do projeto.
**Valida:** se está no Databricks e se origem e destino são containers diferentes.

Modo de validação: RÁPIDA
✅ Rodando no Databricks
✅ Origem (raw) e destino (squad2) separados


## Etapa 2 — Credenciais
**Faz:** busca as 7 chaves no Secret Scope. Ficam só na memória (aparecem como `[REDACTED]` se impressas).
**Valida:** se o cofre existe, se nenhuma chave falta e se o host do SQL tem o formato certo.

✅ Cofre 'mercadata-ecommerce-pedidos' encontrado
✅ 7 credenciais carregadas (valores ocultos)
✅ SQL_HOST no formato do Azure SQL


...

✅ url_origem OK
✅ tabela_delta_existe OK (Bronze já existe)
✅ ler_metadata OK (None para anotação inexistente)


## Etapa 6 — Funções do SQL Server
**Faz:** cria as funções de gravar e consultar o banco. Gravação pelo conector `sqlserver` (o `jdbc` é bloqueado no Serverless); leitura pelo `jdbc`.
**Valida:** se toda tabela recebe o schema `squad2`.

✅ Tabelas SQL vão para o schema 'squad2'


## Etapa 7 — Teste completo (só ao rodar este notebook direto)
**Faz e valida:** varre o `raw` · lê o micro-lote mais recente e confere colunas e tipos (regra 1) · grava e relê em `metadata/` · `SELECT 1` no SQL Server · confere se o schema `squad2` existe no banco.
No modo rápido (`%run`), é pulada. No fim, limpa as variáveis temporárias.

Teste completo pulado (modo rápido).
⚙️ 00_setup_config pronto para uso


In [0]:
from pyspark.sql import functions as F

FUSO_HORAS = int(((ler_metadata("eda_raw") or {}).get("fuso") or {}).get("horas_ate_utc", 0))   # janela do gerador -> UTC
ANOTACAO_MARCO = "teste_tempo_real"
PADRAO_JANELA = r"(\d{4}/\d{2}/\d{2}/\d{6})"

# ------------------------------ validação ------------------------------
checar(tabela_delta_existe(CAMINHO_BRONZE), "Bronze encontrada", "Bronze não existe: rode o 03_ingestao_bronze antes")
print(f"Fuso do gerador: somar {FUSO_HORAS} h para UTC")

✅ Bronze encontrada
Fuso do gerador: somar 3 h para UTC


## Etapa 2 — Marco zero (rodar ANTES do orquestrador)
**Faz:** grava o horário de início do teste e a última ingestão da Bronze. Tudo o que entrar na Bronze depois disso é "novo" para o teste.
Rodar de novo reinicia o teste.
**Valida:** releitura do marco.

In [0]:
ultima_ingestao = ler_delta(CAMINHO_BRONZE).agg(F.max("bronze_ingested_at")).first()[0]
marco = {"inicio_utc": datetime.now(timezone.utc).isoformat(), "bronze_ultima_ingestao": str(ultima_ingestao) if ultima_ingestao else None}
escrever_metadata(ANOTACAO_MARCO, marco)

# ------------------------------ validação ------------------------------
checar((ler_metadata(ANOTACAO_MARCO) or {}).get("inicio_utc") == marco["inicio_utc"], f"Marco zero gravado: {marco['inicio_utc']}",
       "O marco zero não foi gravado")
print("Agora rode o 06_orquestrador durante a janela de tempo real; quando ele parar, volte aqui e rode as Etapas 3 a 6.")

✅ Marco zero gravado: 2026-10-08T17:31:31.952792+00:00
Agora rode o 06_orquestrador durante a janela de tempo real; quando ele parar, volte aqui e rode as Etapas 3 a 6.


## Etapa 3 — Ciclos do orquestrador
**Faz:** lê os registros do orquestrador iniciados depois do marco e separa os ciclos em que a Bronze gravou arquivos novos.
**Valida:** existe ao menos um ciclo em que a Bronze recebeu arquivo **e**, no mesmo ciclo, Silver e Gold terminaram com sucesso, com a Silver processando ao menos as linhas que a Bronze trouxe.

In [0]:
marco = ler_metadata(ANOTACAO_MARCO)
checar(marco is not None, f"Marco zero encontrado ({marco and marco['inicio_utc']})", "Sem marco zero: rode a Etapa 2 antes do orquestrador")
inicio_teste = pd.Timestamp(marco["inicio_utc"]).tz_convert(None).floor("s")

# Registros orquestrador_AAAAMMDD_HHMMSS.json (UTC) iniciados depois do marco
nomes = sorted(p.name.rsplit("/", 1)[1][:-5] for p in fs_squad.get_paths(path=f"metadata/{TABELA}", recursive=False)
               if p.name.rsplit("/", 1)[1].startswith("orquestrador_"))
execucoes = [n for n in nomes if pd.to_datetime(n.split("_", 1)[1], format="%Y%m%d_%H%M%S") >= inicio_teste]

linhas = []
for execucao in execucoes:
    for r in (ler_metadata(execucao) or {}).get("ciclos", []):
        b, s, g = (r.get(n, {}) for n in ("./03_ingestao_bronze", "./04_ingestao_silver", "./05_ingestao_gold"))
        linhas.append({"execucao": execucao, "ciclo": r["ciclo"], "inicio_utc": r["inicio_utc"],
                       "bronze_arquivos": b.get("resumo", {}).get("arquivos_gravados", 0), "bronze_linhas": b.get("resumo", {}).get("linhas_gravadas", 0),
                       "silver": s.get("status", "—"), "silver_linhas": s.get("resumo", {}).get("aprovadas", 0) + s.get("resumo", {}).get("quarentena", 0),
                       "gold": g.get("status", "—"), "gold_alertas_novos": g.get("resumo", {}).get("alertas_novos", 0)})
ciclos = pd.DataFrame(linhas, columns=["execucao", "ciclo", "inicio_utc", "bronze_arquivos", "bronze_linhas", "silver", "silver_linhas", "gold", "gold_alertas_novos"])
com_dados = ciclos[ciclos["bronze_arquivos"] > 0]
atravessaram = com_dados[(com_dados["silver"] == "OK") & (com_dados["gold"] == "OK") & (com_dados["silver_linhas"] >= com_dados["bronze_linhas"])]

# ------------------------------ validação ------------------------------
checar(len(execucoes) > 0, f"{len(execucoes)} execução(ões) do orquestrador depois do marco", "Nenhuma execução do orquestrador depois do marco", parar=False)
checar(len(com_dados) > 0, f"{len(com_dados)} ciclos receberam arquivos novos", "Nenhum ciclo recebeu arquivo novo: o gerador estava ligado?", parar=False)
checar(len(atravessaram) > 0, f"{len(atravessaram)} ciclos levaram os arquivos novos até a Gold no mesmo ciclo",
       "Nenhum ciclo levou arquivo novo até a Gold no mesmo ciclo", parar=False)

# ------------------------------ resultado ------------------------------
display(ciclos) if len(ciclos) else print("Sem ciclos registrados depois do marco.")

✅ Marco zero encontrado (2026-10-07T12:24:18.568052+00:00)
✅ 1 execução(ões) do orquestrador depois do marco
✅ 25 ciclos receberam arquivos novos
✅ 25 ciclos levaram os arquivos novos até a Gold no mesmo ciclo


execucao,ciclo,inicio_utc,bronze_arquivos,bronze_linhas,silver,silver_linhas,gold,gold_alertas_novos
orquestrador_20261007_122700,1,2026-10-07T12:27:01+00:00,6,25,OK,25,OK,12
orquestrador_20261007_122700,2,2026-10-07T12:31:26+00:00,1,20,OK,20,OK,1
orquestrador_20261007_122700,3,2026-10-07T12:35:00+00:00,1,20,OK,20,OK,1
orquestrador_20261007_122700,4,2026-10-07T12:38:44+00:00,2,18,OK,18,OK,1
orquestrador_20261007_122700,5,2026-10-07T12:42:18+00:00,2,11,OK,11,OK,2
orquestrador_20261007_122700,6,2026-10-07T12:45:52+00:00,1,14,OK,14,OK,1
orquestrador_20261007_122700,7,2026-10-07T12:49:26+00:00,2,7,OK,7,OK,0
orquestrador_20261007_122700,8,2026-10-07T12:52:50+00:00,2,6,OK,6,OK,0
orquestrador_20261007_122700,9,2026-10-07T12:56:14+00:00,1,5,OK,5,OK,0
orquestrador_20261007_122700,10,2026-10-07T12:59:39+00:00,1,6,OK,6,OK,1


## Etapa 4 — Rastreio de cada arquivo novo
**Faz:** para cada micro-lote que entrou na Bronze depois do marco: linhas na Bronze, na Silver e na quarentena, e os horários de cada passagem.
A latência parte da **janela do gerador** (o horário no nome da pasta, convertido para UTC).
**Valida:** Bronze = Silver + quarentena em todos os arquivos (nenhum pedido ficou pelo caminho).

In [0]:
bronze_novo = ler_delta(CAMINHO_BRONZE)
if marco["bronze_ultima_ingestao"]:
    bronze_novo = bronze_novo.filter(F.col("bronze_ingested_at") > F.to_timestamp(F.lit(marco["bronze_ultima_ingestao"])))
por_bronze = bronze_novo.groupBy("bronze_source_file").agg(F.count("*").alias("linhas_bronze"), F.max("bronze_ingested_at").alias("bronze_utc"))
arquivos_novos = [r[0] for r in por_bronze.select("bronze_source_file").collect()]


def por_destino(caminho, rotulo):
    """Linhas e horário de processamento de cada arquivo novo num destino da Silver."""
    if not arquivos_novos or not tabela_delta_existe(caminho):
        return None
    return (ler_delta(caminho).filter(F.col("bronze_source_file").isin(arquivos_novos)).groupBy("bronze_source_file")
            .agg(F.count("*").alias(f"linhas_{rotulo}"), F.max("silver_processed_at").alias(f"{rotulo}_utc")))


rastreio = por_bronze
for destino in [por_destino(CAMINHO_SILVER, "silver"), por_destino(CAMINHO_QUARANTINE, "quarentena")]:
    if destino is not None:
        rastreio = rastreio.join(destino, "bronze_source_file", "left")
rastreio = rastreio.toPandas()
COLUNAS_RASTREIO = ["bronze_source_file", "janela_utc", "bronze_utc", "processado_utc", "linhas_bronze", "linhas_silver",
                    "linhas_quarentena", "latencia_ate_bronze_s", "latencia_ate_silver_s", "completo"]
if len(rastreio):
    for coluna in ["linhas_silver", "linhas_quarentena"]:
        rastreio[coluna] = rastreio.get(coluna, pd.Series(0, index=rastreio.index)).fillna(0).astype(int)
    horarios = [c for c in ["silver_utc", "quarentena_utc"] if c in rastreio]
    rastreio["bronze_utc"] = pd.to_datetime(rastreio["bronze_utc"])                       # garante tipo data antes das contas
    rastreio["processado_utc"] = rastreio[horarios].apply(pd.to_datetime).max(axis=1) if horarios else pd.NaT
    rastreio["janela_utc"] = (pd.to_datetime(rastreio["bronze_source_file"].str.extract(PADRAO_JANELA)[0], format="%Y/%m/%d/%H%M%S")
                              + pd.Timedelta(hours=FUSO_HORAS))
    rastreio["latencia_ate_bronze_s"] = (rastreio["bronze_utc"] - rastreio["janela_utc"]).dt.total_seconds().round()
    rastreio["latencia_ate_silver_s"] = (rastreio["processado_utc"] - rastreio["janela_utc"]).dt.total_seconds().round()
    rastreio["completo"] = rastreio["linhas_bronze"] == rastreio["linhas_silver"] + rastreio["linhas_quarentena"]
    rastreio = rastreio.sort_values("janela_utc")[COLUNAS_RASTREIO]
else:
    rastreio = pd.DataFrame(columns=COLUNAS_RASTREIO)        # sem arquivo novo: tabela vazia, mas com as colunas que as Etapas 5 e 6 usam

# ------------------------------ validação ------------------------------
checar(len(rastreio) > 0, f"{len(rastreio)} arquivos novos na Bronze depois do marco", "Nenhum arquivo novo na Bronze depois do marco", parar=False)
if len(rastreio):
    checar(rastreio["completo"].all(), "Todos os arquivos chegaram inteiros à Silver ou à quarentena",
           f"Arquivos incompletos: {rastreio.loc[~rastreio['completo'], 'bronze_source_file'].tolist()}", parar=False)

# ------------------------------ resultado ------------------------------
display(rastreio) if len(rastreio) else print("Nenhum arquivo novo para rastrear.")   # display() não aceita tabela vazia

✅ 38 arquivos novos na Bronze depois do marco
✅ Todos os arquivos chegaram inteiros à Silver ou à quarentena


bronze_source_file,janela_utc,bronze_utc,processado_utc,linhas_bronze,linhas_silver,linhas_quarentena,latencia_ate_bronze_s,latencia_ate_silver_s,completo
real-time-data/2026/10/07/091526/ecommerce_pedidos.parquet,2026-10-07T12:15:26.000Z,2026-10-07T12:27:43.308Z,2026-10-07T12:28:20.528Z,6,6,0,737.0,775.0,true
real-time-data/2026/10/07/091602/ecommerce_pedidos.parquet,2026-10-07T12:16:02.000Z,2026-10-07T12:27:43.308Z,2026-10-07T12:28:20.528Z,6,6,0,701.0,739.0,true
real-time-data/2026/10/07/092003/ecommerce_pedidos.parquet,2026-10-07T12:20:03.000Z,2026-10-07T12:27:43.308Z,2026-10-07T12:28:20.528Z,3,3,0,460.0,498.0,true
real-time-data/2026/10/07/092203/ecommerce_pedidos.parquet,2026-10-07T12:22:03.000Z,2026-10-07T12:27:43.308Z,2026-10-07T12:28:23.905Z,6,4,2,340.0,381.0,true
real-time-data/2026/10/07/092403/ecommerce_pedidos.parquet,2026-10-07T12:24:03.000Z,2026-10-07T12:27:43.308Z,2026-10-07T12:28:20.528Z,2,2,0,220.0,258.0,true
real-time-data/2026/10/07/092602/ecommerce_pedidos.parquet,2026-10-07T12:26:02.000Z,2026-10-07T12:27:43.308Z,2026-10-07T12:28:23.905Z,2,1,1,101.0,142.0,true
real-time-data/2026/10/07/092803/ecommerce_pedidos.parquet,2026-10-07T12:28:03.000Z,2026-10-07T12:31:45.620Z,2026-10-07T12:32:16.243Z,20,17,3,223.0,253.0,true
real-time-data/2026/10/07/093203/ecommerce_pedidos.parquet,2026-10-07T12:32:03.000Z,2026-10-07T12:35:18.188Z,2026-10-07T12:35:49.408Z,20,19,1,195.0,226.0,true
real-time-data/2026/10/07/093603/ecommerce_pedidos.parquet,2026-10-07T12:36:03.000Z,2026-10-07T12:39:00.948Z,2026-10-07T12:39:32.303Z,10,9,1,178.0,209.0,true
real-time-data/2026/10/07/093803/ecommerce_pedidos.parquet,2026-10-07T12:38:03.000Z,2026-10-07T12:39:00.948Z,2026-10-07T12:39:29.885Z,8,8,0,58.0,87.0,true


## Etapa 5 — A Gold depois da chegada
**Faz:** compara o último ciclo da Gold (anotado em `metadata/gold_controle.json`) com o processamento dos arquivos novos na Silver,
e o pedido mais recente que a Gold considerou com o pedido mais recente desses arquivos.
**Valida:** a Gold rodou depois da Silver e já incorpora o pedido mais novo do teste.

In [0]:
controle_gold = ler_metadata("gold_controle") or {}
ciclo_gold = pd.to_datetime(str(controle_gold.get("ultimo_ciclo_id", "")), format="%Y%m%d%H%M%S", errors="coerce")
referencia_gold = pd.to_datetime(controle_gold.get("referencia"), errors="coerce")             # pedido mais recente (Brasília)
ultimo_processado = rastreio["processado_utc"].max() if len(rastreio) else pd.NaT
pedido_mais_novo = (ler_delta(CAMINHO_SILVER).filter(F.col("bronze_source_file").isin(arquivos_novos))
                    .agg(F.max(F.col("dt_pedido") - F.expr("INTERVAL 3 HOURS"))).first()[0] if arquivos_novos else None)  # UTC -> Brasília

# ------------------------------ validação ------------------------------
checar(pd.notna(ciclo_gold), f"Último ciclo da Gold: {ciclo_gold} UTC", "Nenhum ciclo da Gold registrado", parar=False)
if pd.notna(ciclo_gold) and pd.notna(ultimo_processado):
    checar(ciclo_gold >= ultimo_processado.floor("s"), "A Gold recalculou depois da chegada dos arquivos novos",
           "A Gold não rodou depois da chegada dos arquivos novos", parar=False)
if pedido_mais_novo is not None and pd.notna(referencia_gold):
    checar(referencia_gold >= pd.Timestamp(pedido_mais_novo), f"A Gold já considera o pedido mais novo do teste ({pd.Timestamp(pedido_mais_novo)})",
           "A Gold ainda não considera o pedido mais novo do teste", parar=False)

✅ Último ciclo da Gold: 2026-10-07 13:57:57 UTC
✅ A Gold recalculou depois da chegada dos arquivos novos
✅ A Gold já considera o pedido mais novo do teste (2026-10-07 10:55:55.051961)


## Etapa 6 — Veredito
**Faz:** junta as provas das etapas anteriores e calcula a latência de ponta a ponta (janela do gerador → indicadores da Gold).

In [0]:
latencia_gold_s = ((ciclo_gold - rastreio["janela_utc"].max()).total_seconds() if len(rastreio) and pd.notna(ciclo_gold) else None)
provas = pd.DataFrame([
    ("Ciclo com Bronze → Silver → Gold no mesmo ciclo", f"{len(atravessaram)} ciclos", len(atravessaram) > 0),
    ("Arquivos novos rastreados", f"{len(rastreio)} arquivos", len(rastreio) > 0),
    ("Nenhum pedido perdido (Bronze = Silver + quarentena)", f"{int(rastreio['completo'].sum()) if len(rastreio) else 0} de {len(rastreio)}",
     bool(len(rastreio)) and bool(rastreio["completo"].all())),
    ("Gold recalculada depois da chegada", str(ciclo_gold), pd.notna(ciclo_gold) and pd.notna(ultimo_processado) and ciclo_gold >= ultimo_processado.floor("s")),
    ("Latência janela do gerador → Gold (último arquivo)", "—" if latencia_gold_s is None else f"{latencia_gold_s / 60:.1f} min", latencia_gold_s is not None),
], columns=["prova", "resultado", "ok"])
provas["ok"] = provas["ok"].map({True: "✅", False: "⚠️"})

# ------------------------------ validação ------------------------------
aprovado = (provas["ok"] == "✅").all()
checar(aprovado, "TESTE APROVADO: arquivos novos atravessaram Bronze → Silver → Gold no mesmo ciclo do orquestrador",
       "Teste incompleto: veja as provas com ⚠️", parar=False)

# ------------------------------ resultado ------------------------------
display(provas)

✅ TESTE APROVADO: arquivos novos atravessaram Bronze → Silver → Gold no mesmo ciclo do orquestrador


prova,resultado,ok
Ciclo com Bronze → Silver → Gold no mesmo ciclo,25 ciclos,✅
Arquivos novos rastreados,38 arquivos,✅
Nenhum pedido perdido (Bronze = Silver + quarentena),38 de 38,✅
Gold recalculada depois da chegada,2026-10-07 13:57:57,✅
Latência janela do gerador → Gold (último arquivo),1.8 min,✅
